In [2]:
RESULTS_CSV    = 'results.csv'
SCORECARD_XLSX = 'scorecard.xlsx'
INDEX_HTML     = 'index.html'

# defaults by score window, we should probably do special summaries for the ones that require in person, were uniquely weird, et cetera.
SUMMARY_BANDS = [
    ( 3.5,  '{town} responded to most requests quickly and in full.'),
    ( 2,  '{town} was broadly cooperative on public records requests.'),
    (-0.5,  '{town} shows mixed compliance with the state\'s right-to-know law.'),
    (-2.5,  '{town} is slow to respond and frequently rejects records routinely considered public.'),
    (-99,   '{town} is largely unresponsive to public records requests filed under the state\'s right-to-know law.'),
]

def default_summary(town: str, score):
    if score is None: return f'{town} has no graded responses yet.'
    for threshold, template in SUMMARY_BANDS:
        if score >= threshold:
            return template.format(town=town)
    return SUMMARY_BANDS[-1][1].format(town=town)

In [3]:
import csv, re, json
from collections import defaultdict, Counter
from datetime import date, datetime
from openpyxl import load_workbook

ACK_DEADLINE_BD       = 5
FULFILL_DEADLINE_BD   = 25
IMMEDIATE_BD          = 5
DELAYED_FULFILLED_BD  = 60
AUTO_EXCLUDE_CATEGORIES   = {'RFP List'}
AUTO_EXCLUDE_STATUS_CODES = {'fix'}

JURISDICTION_REMAP = {
    'Berlin School District/SAU 3':'Berlin', 'SAU6, serving the communities of Claremont and Unity':'Claremont',
    'Concord School District/SAU8':'Concord', 'Dover School District/SAU11':'Dover',
    'Franklin School District/SAU18':'Franklin', 'Manchester School District':'Manchester',
    'Keene School District/SAU29':'Keene', 'Laconia School District/SAU30':'Laconia',
    'Lebanon School District/SAU88':'Lebanon', 'Nashua School District':'Nashua',
    'Portsmouth School District':'Portsmouth', 'Rochester School District':'Rochester',
    'Somersworth School District/SAU56':'Somersworth', 'Contoocook Valley School District/SAU1':'Peterborough',
    'Shaker Regional School District/SAU80':'Belmont', 'SAU70':'Hanover', 'SAU20':'Gorham', 'SAU67':'Bow',
    'Hopkinton School District':'Hopkinton', 'Derry Cooperative School District':'Derry',
    'Salem School District/SAU57':'Salem', 'Monadnock Regional School District':'Troy',
    'Oyster River Cooperative School District':'Durham',
}
AGENCY_REMAP = {
    ('Lebanon','City Clerk'):'Lebanon City Clerk', ('Lebanon','Lebanon Clerk'):'Lebanon City Clerk',
    ('Nashua','Nashua City Clerk'):'Nashua City Clerk', ('Nashua','Nashua Clerk'):'Nashua City Clerk',
}

_TRAIL_PAREN = re.compile(r'\s*\([^)]*\)\s*$')
def category(t): return _TRAIL_PAREN.sub('', t).strip()
def resolved_jurisdiction(r): return JURISDICTION_REMAP.get(r['Agency Name'], r['Jurisdiction'])
def resolved_agency(r): return AGENCY_REMAP.get((resolved_jurisdiction(r), r['Agency Name']), r['Agency Name'])
def normalized_tags(r): return (r.get('Tags') or '').lower().replace('nh-in-person','nh-inperson')
def has_tag(r, frag): return frag in normalized_tags(r)
def _parse_dt(s):
    if not s: return None
    try: return datetime.strptime(s.split()[0], '%Y-%m-%d').date()
    except: return None
def business_days_between(a, b):
    if not a or not b or b <= a: return 0
    total = (b-a).days; w, r = divmod(total, 7); n = w*5; wd = a.weekday()
    for d in range(1, r+1):
        if (wd+d)%7 < 5: n += 1
    return n
def business_days_open(r, today=None):
    s = _parse_dt(r.get('Date Submitted')); d = _parse_dt(r.get('Date Done'))
    return business_days_between(s, d or (today or date.today()))

def auto_score(r, today=None):
    code = (r.get('Status Code') or '').strip(); bd = business_days_open(r, today)
    if has_tag(r, 'privacy'): return -2
    if code == 'done':
        if has_tag(r, 'nh-inperson') or has_tag(r, 'nh-citizen'): return 2
        if bd > DELAYED_FULFILLED_BD: return 3
        if bd <= IMMEDIATE_BD: return 5
        return 4
    if code == 'no_docs':  return 0
    if code == 'fix':      return None
    if code == 'partial':  return None
    if code == 'rejected': return -4
    if code == 'ack':                          return -5 if bd > ACK_DEADLINE_BD else None
    if code in ('processed','submitted'):      return -3 if bd > FULFILL_DEADLINE_BD else None
    return None

In [4]:
with open(RESULTS_CSV, newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))

wb = load_workbook(SCORECARD_XLSX, data_only=False)
ag_ws    = wb['Agencies']
juris_ws = wb['Jurisdictions']

header_row = [c.value for c in ag_ws[1]]
categories  = []
for h in header_row[3:]:
    if h and ' — Status' in h:
        categories.append(h.replace(' — Status', ''))
    elif h and ' — Exclude' in h:
        break
N = len(categories)
STATUS_START  = 4
EXCLUDE_START = STATUS_START + N
AUTO_START    = EXCLUDE_START + N
MANUAL_START  = AUTO_START + N

ag_overrides = {} 
for r in ag_ws.iter_rows(min_row=2, values_only=False):
    j = r[0].value; a = r[1].value
    if not j: continue
    per_cat = {}
    for i, cat in enumerate(categories):
        excl = bool(r[EXCLUDE_START-1+i].value)
        man  = r[MANUAL_START-1+i].value
        per_cat[cat] = {'exclude': excl, 'manual': man if isinstance(man, (int, float)) else None}
    ag_overrides[(j, a)] = per_cat

juris_summary = {}
for r in juris_ws.iter_rows(min_row=2, values_only=True):
    j = r[0]; summary = (r[4] if len(r) > 4 else None)
    if j: juris_summary[j] = (summary or '').strip()

print(f'Loaded {len(rows)} requests, {len(ag_overrides)} agency rows, {len(categories)} categories.')
print(f'Jurisdictions with non-blank Summary: {sum(1 for v in juris_summary.values() if v)}')

Loaded 280 requests, 69 agency rows, 11 categories.
Jurisdictions with non-blank Summary: 0


In [5]:
today = date.today()

# Pivot rows by (juris, agency) → cat → [requests]
by_ag = defaultdict(lambda: defaultdict(list))
for r in rows:
    by_ag[(resolved_jurisdiction(r), resolved_agency(r))][category(r['Title'])].append(r)

def effective_score(req, override):
    if override.get('manual') is not None:
        return float(override['manual'])
    return auto_score(req, today)

town_stats = {}
for j in sorted({jj for jj, _ in by_ag}):
    agency_avgs = []
    j_reqs = []                      
    scored_count = 0
    for (jj, a), cat_map in by_ag.items():
        if jj != j: continue
        cell_scores = []
        ov_map = ag_overrides.get((j, a), {})
        for cat in categories:
            reqs = cat_map.get(cat, [])
            ov = ov_map.get(cat, {'exclude': cat in AUTO_EXCLUDE_CATEGORIES, 'manual': None})
            if not reqs:
                continue
            req = sorted(reqs, key=lambda r: r.get('Date Submitted') or '', reverse=True)[0]
            j_reqs.append(req)
            if ov.get('exclude'):
                continue
            s = effective_score(req, ov)
            if s is None: continue
            cell_scores.append(s)
            scored_count += 1
        if cell_scores:
            agency_avgs.append(sum(cell_scores) / len(cell_scores))
    score = round(sum(agency_avgs) / len(agency_avgs), 2) if agency_avgs else None

    total = len(j_reqs)
    responded_codes = {'done', 'no_docs', 'partial', 'rejected', 'fix'}
    responded = sum(1 for r in j_reqs if (r.get('Status Code') or '').strip() in responded_codes)
    rejected  = sum(1 for r in j_reqs if (r.get('Status Code') or '').strip() == 'rejected')
    done_reqs = [r for r in j_reqs if (r.get('Status Code') or '').strip() == 'done']
    if any(has_tag(r, 'nh-inperson') or has_tag(r, 'nh-citizen') for r in done_reqs):
        records = 'in-person'
    elif done_reqs:
        records = 'digital'
    else:
        records = 'none'
    reply_bds = [business_days_open(r, today) for r in j_reqs
                  if (r.get('Status Code') or '').strip() in responded_codes
                  and business_days_open(r, today) > 0]
    avg_reply = f'{round(sum(reply_bds)/len(reply_bds))} days' if reply_bds else '—'
    missed = any(
        (r.get('Status Code') or '').strip() == 'ack'  and business_days_open(r, today) > ACK_DEADLINE_BD
        or (r.get('Status Code') or '').strip() in ('processed','submitted') and business_days_open(r, today) > FULFILL_DEADLINE_BD
        for r in j_reqs
    )
    summary = juris_summary.get(j) or default_summary(j, score)

    town_stats[j] = {
        'score':     score if score is not None else 0,
        'requests':  total,
        'responded': responded,
        'graded':    scored_count,
        'avgReply':  avg_reply,
        'records':   records,
        'rejected':  rejected,
        'missedDeadline': bool(missed),
        'summary':   summary,
    }

# Preview
for j in sorted(town_stats, key=lambda j: -town_stats[j]['score']):
    s = town_stats[j]
    print(f'{j:<14} score={s["score"]:+5.2f}  req={s["requests"]:>2}  resp={s["responded"]:>2}  graded={s["graded"]:>2}  rej={s["rejected"]:>2}  reply={s["avgReply"]:<8}  rec={s["records"]:<9}  miss={s["missedDeadline"]}')

Durham         score=+3.50  req=12  resp= 6  graded= 6  rej= 0  reply=8 days    rec=digital    miss=False
Derry          score=+1.94  req=12  resp=10  graded=12  rej= 0  reply=10 days   rec=digital    miss=True
Belmont        score=+1.83  req=12  resp=11  graded= 9  rej= 0  reply=24 days   rec=in-person  miss=True
Troy           score=+1.38  req=12  resp= 7  graded= 8  rej= 1  reply=17 days   rec=digital    miss=True
Hanover        score=+1.35  req=12  resp=11  graded=11  rej= 2  reply=11 days   rec=in-person  miss=False
Laconia        score=+1.32  req=13  resp=13  graded=13  rej= 2  reply=9 days    rec=in-person  miss=False
Franklin       score=+1.31  req=12  resp= 8  graded= 9  rej= 1  reply=18 days   rec=digital    miss=True
Hopkinton      score=+0.75  req=12  resp=12  graded=10  rej= 1  reply=18 days   rec=digital    miss=False
Keene          score=+0.75  req=12  resp= 7  graded=12  rej= 0  reply=19 days   rec=digital    miss=True
Manchester     score=+0.63  req=12  resp=10  graded

In [6]:
def js_str(s):
    return json.dumps(s, ensure_ascii=False)

def js_bool(b): return 'true' if b else 'false'

lines = ['const TOWN_OVERRIDES = {']
for j in sorted(town_stats):
    s = town_stats[j]
    lines.append(f'  {js_str(j)}: {{')
    lines.append(f'    score:          {s["score"]},')
    lines.append(f'    requests:       {s["requests"]},')
    lines.append(f'    responded:      {s["responded"]},')
    lines.append(f'    graded:         {s["graded"]},')
    lines.append(f'    avgReply:       {js_str(s["avgReply"])},')
    lines.append(f'    records:        {js_str(s["records"])},')
    lines.append(f'    rejected:       {s["rejected"]},')
    lines.append(f'    missedDeadline: {js_bool(s["missedDeadline"])},')
    lines.append(f'    summary:        {js_str(s["summary"])},')
    lines.append(f'  }},')
lines.append('};')
new_block = '\n'.join(lines)
print(new_block[:1200] + '\n…')

const TOWN_OVERRIDES = {
  "Belmont": {
    score:          1.83,
    requests:       12,
    responded:      11,
    graded:         9,
    avgReply:       "24 days",
    records:        "in-person",
    rejected:       0,
    missedDeadline: true,
    summary:        "Belmont shows mixed compliance with the state's right-to-know law.",
  },
  "Berlin": {
    score:          -1.58,
    requests:       12,
    responded:      8,
    graded:         10,
    avgReply:       "11 days",
    records:        "in-person",
    rejected:       1,
    missedDeadline: true,
    summary:        "Berlin is slow to respond and frequently rejects records routinely considered public.",
  },
  "Bow": {
    score:          -1.39,
    requests:       12,
    responded:      4,
    graded:         9,
    avgReply:       "43 days",
    records:        "digital",
    rejected:       0,
    missedDeadline: true,
    summary:        "Bow is slow to respond and frequently rejects records routinely considered p

In [8]:
with open(INDEX_HTML, 'r', encoding='utf-8') as f:
    html = f.read()

pattern = re.compile(r'const TOWN_OVERRIDES = \{[\s\S]*?\n\};', re.MULTILINE)
if not pattern.search(html):
    raise RuntimeError('Could not find existing `const TOWN_OVERRIDES = { ... };` block in index.html')

new_html, n = pattern.subn(new_block, html, count=1)
with open(INDEX_HTML, 'w', encoding='utf-8') as f:
    f.write(new_html)
print(f'updated {INDEX_HTML} ({n} replacement, {len(town_stats)} towns)')

updated index.html (1 replacement, 23 towns)
